# 🎾 Stage 1 — Player Detection

First stage of the **Padel Match Analytics** pipeline: detect the players in each frame with **RF-DETR** (Roboflow's transformer detector — no NMS, strong on overlapping people).

This notebook is the narrative companion to the code in [`src/padel_analytics`](../src/padel_analytics). It:
1. loads a frame from the match video,
2. runs the detector on a single frame,
3. sweeps a sample of frames and looks at detection statistics.

> The ball is **not** detected here — it's a tiny, fast object that needs a dedicated model (a later stage). The detector lives behind a swappable `ObjectDetector` interface, so RF-DETR and YOLO are interchangeable via `build_detector(config)`.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from padel_analytics.config import Config
from padel_analytics.detection import DetectionAnnotator, build_detector
from padel_analytics.detection.classes import PLAYER
from padel_analytics.video.io import frame_generator, video_info

plt.rcParams["figure.dpi"] = 110

VIDEO = Path("../data/raw/padel_clip.mp4")
info = video_info(VIDEO)
print(info)

## 1. Peek at a raw frame

In [ ]:
frame = next(frame_generator(VIDEO, stride=1))

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Raw frame")
plt.show()

## 2. Detect players on a single frame

The default config uses RF-DETR (`medium`). The first call downloads the pretrained weights.

In [ ]:
detector = build_detector(Config().detector)  # RF-DETR medium by default
annotator = DetectionAnnotator()

detections = detector.detect(frame)
print(f"{len(detections)} players detected")

annotated = annotator.annotate(frame, detections)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("RF-DETR player detections")
plt.show()

## 3. Detection statistics across the match

Sample every 30th frame and count detected players. We expect ~4 on-court players, plus
the occasional spectator/referee picked up around the court (a later court-region filter
will isolate the four active players).

In [ ]:
records = []
for i, fr in enumerate(frame_generator(VIDEO, stride=30)):
    det = detector.detect(fr)
    records.append({"frame_idx": i * 30, "players": int((det.class_id == PLAYER).sum())})

df = pd.DataFrame(records)
print(f"Sampled {len(df)} frames")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["players"].value_counts().sort_index().plot.bar(ax=axes[0], color="#0b5394")
axes[0].set_title("Players detected per frame")
axes[0].set_xlabel("# players")
axes[0].set_ylabel("frames")

df.set_index("frame_idx")["players"].plot(ax=axes[1], color="#0b5394")
axes[1].axhline(4, color="#e06666", ls="--", label="4 on-court players")
axes[1].set_title("Players detected over time")
axes[1].set_xlabel("frame")
axes[1].set_ylabel("# players")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Avg players/frame: {df.players.mean():.2f}  (min {df.players.min()}, max {df.players.max()})")

## Takeaways & next steps

- **Players** are detected reliably by RF-DETR. Counts above 4 are background people (spectators / staff) around the court.
- Detections are **per-frame and identity-less** — no notion of *which* player is which across frames.

**Stage 2** adds multi-object **tracking** (ByteTrack) to give each player a persistent ID, and a **court-region filter** to keep only the four active players. → see the roadmap in the [README](../README.md).